# TOM Value Analysis: What distinguishes persuasive comments?

**Setup**: For each CMV post, we have:
- `persuasive`: delta-awarded comment (changed OP's mind)
- `hard_negative`: top-level comment that didn't get a delta (tried, failed)
- `easy_negative`: random comment

Each has TOM values extracted across 6 categories: beliefs, desires, intentions, emotions, knowledge, perspective_taking.

**Question**: Do persuasive comments align more (or differently) with the post's TOM values?

## Method: Per-Category Token-Level Analysis

Instead of embedding full value strings or pooling all categories together, we:
1. **Parse** each category's comma-separated values into individual tokens
2. **Embed** each token with `all-MiniLM-L6-v2`
3. **Mean-pool per category** to get one vector per post/comment per category
4. **Compute 5 feature types per category**: cosine similarity, Jaccard overlap, value novelty, value coverage, token count

This yields **31 features** (6 categories x 5 features + 1 global sim) per post-comment pair.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.utils.class_weight import compute_sample_weight
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

CATEGORIES = ['beliefs', 'desires', 'intentions', 'emotions', 'knowledge', 'perspective_taking']
# Resolve repo root whether the notebook cwd is repo root or scripts/analysis/
REPO_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data' / 'tom_pairs.json').exists())
PAIRS_PATH = REPO_ROOT / 'data' / 'tom_pairs.json'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

type_order = ['persuasive', 'hard_negative', 'easy_negative']
palette = {'persuasive': '#2ecc71', 'hard_negative': '#e74c3c', 'easy_negative': '#95a5a6'}

## 1. Load data

In [ ]:
with open(PAIRS_PATH) as f:
    data = json.load(f)

pairs = data['pairs']
meta = data['metadata']
print(f"Total pairs: {meta['total_pairs']}")
print(f"Type breakdown: {meta['type_counts']}")

df_raw = pd.DataFrame(pairs)
print(f"\nDataFrame shape: {df_raw.shape}")
df_raw.head(2)

## 2. Parse & Embed Individual Value Tokens

Split each category's comma-separated value string into individual tokens,
embed all unique tokens once, then use these embeddings for all downstream features.

In [ ]:
print('Loading sentence-transformers model...')
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Model loaded.')

In [ ]:
def parse_values(val_str):
    """Split 'Empathy, Nuance, Accountability' into ['empathy', 'nuance', 'accountability']."""
    if not val_str:
        return []
    return [v.strip().lower() for v in val_str.split(',') if v.strip()]

# Collect all unique individual value tokens across all pairs
all_tokens = set()
for pair in pairs:
    for side in ['post_values', 'comment_values']:
        for cat in CATEGORIES:
            all_tokens.update(parse_values(pair[side].get(cat)))

all_tokens = list(all_tokens)
print(f'Unique individual value tokens: {len(all_tokens)}')

print('Embedding all tokens...')
token_embs = model.encode(all_tokens, batch_size=512, show_progress_bar=True, normalize_embeddings=True)
tok_map = {t: token_embs[i] for i, t in enumerate(all_tokens)}
DIM = token_embs.shape[1]
print(f'Done. Embedding dim: {DIM}')

## 3. Compute Per-Category Features

For each post-comment pair and each TOM category, compute:
- **Cosine similarity**: between mean-pooled post and comment token embeddings
- **Jaccard overlap**: |intersection| / |union| of exact token sets
- **Value novelty**: fraction of comment tokens NOT in the post
- **Value coverage**: fraction of post tokens present in the comment
- **Token count**: number of value tokens in the comment

Plus one **global similarity** (all tokens pooled across all categories).

In [ ]:
def mean_pool(tokens):
    """Mean-pool token embeddings into a unit-normalized vector."""
    vecs = [tok_map[t] for t in tokens if t in tok_map]
    if not vecs:
        return None
    agg = np.mean(vecs, axis=0)
    n = np.linalg.norm(agg)
    return agg / n if n > 0 else agg

rows = []
for pair in pairs:
    row = {
        'post_id': pair['post_id'],
        'comment_id': pair['comment_id'],
        'comment_type': pair['comment_type'],
    }
    all_post_vecs, all_comment_vecs = [], []

    for cat in CATEGORIES:
        post_tokens = parse_values(pair['post_values'].get(cat))
        comment_tokens = parse_values(pair['comment_values'].get(cat))
        post_set, comment_set = set(post_tokens), set(comment_tokens)

        # Cosine similarity (per-category, token-level mean pool)
        post_vec = mean_pool(post_tokens)
        comment_vec = mean_pool(comment_tokens)
        row[f'{cat}_sim'] = float(np.dot(post_vec, comment_vec)) if post_vec is not None and comment_vec is not None else np.nan

        # Jaccard overlap
        union, inter = post_set | comment_set, post_set & comment_set
        row[f'{cat}_jaccard'] = len(inter) / len(union) if union else np.nan

        # Novelty: fraction of comment values NOT in post
        row[f'{cat}_novelty'] = len(comment_set - post_set) / len(comment_set) if comment_set else np.nan

        # Coverage: fraction of post values present in comment
        row[f'{cat}_coverage'] = len(inter) / len(post_set) if post_set else np.nan

        # Token count
        row[f'{cat}_count'] = len(comment_tokens)

        # Collect for global sim
        all_post_vecs.extend([tok_map[t] for t in post_tokens if t in tok_map])
        all_comment_vecs.extend([tok_map[t] for t in comment_tokens if t in tok_map])

    # Global aggregated sim
    if all_post_vecs and all_comment_vecs:
        pv = np.mean(all_post_vecs, axis=0); pv /= np.linalg.norm(pv)
        cv = np.mean(all_comment_vecs, axis=0); cv /= np.linalg.norm(cv)
        row['global_sim'] = float(np.dot(pv, cv))
    else:
        row['global_sim'] = np.nan

    rows.append(row)

df = pd.DataFrame(rows)

# Define feature column groups
sim_cols = [f'{c}_sim' for c in CATEGORIES]
jaccard_cols = [f'{c}_jaccard' for c in CATEGORIES]
novelty_cols = [f'{c}_novelty' for c in CATEGORIES]
coverage_cols = [f'{c}_coverage' for c in CATEGORIES]
count_cols = [f'{c}_count' for c in CATEGORIES]
all_feature_cols = sim_cols + jaccard_cols + novelty_cols + coverage_cols + count_cols + ['global_sim']

print(f"Feature matrix: {df.shape[0]} pairs × {len(all_feature_cols)} features")
df.head(3)

## 4. Per-Category Feature Tables by Comment Type

In [ ]:
for name, cols in [('Cosine Similarity', sim_cols), ('Jaccard Overlap', jaccard_cols),
                    ('Value Novelty', novelty_cols), ('Value Coverage', coverage_cols)]:
    print(f"\n{'='*70}\n{name.upper()} by comment type (mean)\n{'='*70}")
    print(df.groupby('comment_type')[cols].mean().round(4).T.to_string())

# Heatmap of per-category cosine similarity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, cols) in zip(axes, [('Cosine Similarity', sim_cols), ('Value Coverage', coverage_cols)]):
    pivot = df.groupby('comment_type')[cols].mean().T
    pivot.index = [c.replace('_sim', '').replace('_coverage', '') for c in cols]
    sns.heatmap(pivot[type_order], annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
    ax.set_title(f'{name} by Category & Type')
    ax.set_ylabel('')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_category_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Statistical Tests

**Mann-Whitney U test**: Non-parametric test comparing rank orderings. A significant p-value means one group tends to have higher values.

**Cohen's d**: Standardized mean difference. |d| < 0.2 = negligible, 0.2–0.5 = small, 0.5–0.8 = medium, > 0.8 = large.

In [ ]:
persuasive = df[df['comment_type'] == 'persuasive']
hard_neg = df[df['comment_type'] == 'hard_negative']
easy_neg = df[df['comment_type'] == 'easy_negative']

def stat_test(col, a, b):
    av, bv = a[col].dropna().values, b[col].dropna().values
    _, p = stats.mannwhitneyu(av, bv, alternative='two-sided')
    pooled_std = np.sqrt((av.std()**2 + bv.std()**2) / 2)
    d = (av.mean() - bv.mean()) / pooled_std if pooled_std > 0 else 0
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    return {'feature': col, 'mean_a': av.mean(), 'mean_b': bv.mean(),
            'cohen_d': d, 'p_value': p, 'sig': sig}

test_cols = sim_cols + jaccard_cols + novelty_cols + coverage_cols

for label, a, b in [('PERSUASIVE vs HARD_NEGATIVE', persuasive, hard_neg),
                     ('PERSUASIVE vs EASY_NEGATIVE', persuasive, easy_neg)]:
    print(f"\n{'='*80}\n{label}\n{'='*80}")
    print(f"{'Feature':<28} {'mean_A':>8} {'mean_B':>8} {'Cohen d':>9} {'p-value':>12} {'sig':>5}")
    print('-' * 75)
    results = []
    for col in test_cols:
        r = stat_test(col, a, b)
        results.append(r)
        print(f"{r['feature']:<28} {r['mean_a']:>8.4f} {r['mean_b']:>8.4f} {r['cohen_d']:>9.4f} {r['p_value']:>12.4e} {r['sig']:>5}")

    if 'hard' in label.lower():
        stats_ph = pd.DataFrame(results)
        stats_ph.to_csv(RESULTS_DIR / 'feature_stats_p_vs_h.csv', index=False)
    else:
        stats_pe = pd.DataFrame(results)
        stats_pe.to_csv(RESULTS_DIR / 'feature_stats_p_vs_e.csv', index=False)

## 6. Distribution Plots

In [ ]:
# Per-category cosine similarity violin plots
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, cat in zip(axes.flat, CATEGORIES):
    col = f'{cat}_sim'
    sns.violinplot(data=df[df[col].notna()], x='comment_type', y=col,
                   order=type_order, palette=palette, ax=ax, inner='box', cut=0)
    ax.set_title(cat.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.set_ylabel('Cosine Sim')
fig.suptitle('Per-Category Token-Level Cosine Similarity', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_category_violin.png', bbox_inches='tight')
plt.show()

# Global similarity violin
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df[df['global_sim'].notna()], x='comment_type', y='global_sim',
               order=type_order, palette=palette, ax=ax, inner='box', cut=0)
ax.set_title('Global TOM Value Similarity (all tokens pooled)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Cosine Similarity')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'similarity_by_type.png', bbox_inches='tight')
plt.show()

## 7. Paired Analysis: Persuasive vs Hard Negative (Same Post)

Controls for post-level variation by comparing comments on the **same** post.

In [ ]:
feature_groups = {
    'Cosine Sim': sim_cols,
    'Jaccard': jaccard_cols,
    'Novelty': novelty_cols,
    'Coverage': coverage_cols,
}

for group_name, cols in feature_groups.items():
    pp = df[df['comment_type'] == 'persuasive'].set_index('post_id')[cols]
    ph = df[df['comment_type'] == 'hard_negative'].groupby('post_id')[cols].mean()
    common = pp.index.intersection(ph.index)
    diff = pp.loc[common] - ph.loc[common]
    print(f"\n{group_name.upper()} — mean diff (persuasive − hard_neg), {len(common)} posts:")
    for col in cols:
        cat = col.split('_')[0]
        d = diff[col]
        frac = float((d > 0).mean())
        print(f"  {cat:<20} Δ={d.mean():+.5f}  frac_P>H={frac:.3f}")

# Histogram of global_sim paired difference
pp_g = df[df['comment_type'] == 'persuasive'].set_index('post_id')['global_sim']
ph_g = df[df['comment_type'] == 'hard_negative'].groupby('post_id')['global_sim'].mean()
common_g = pp_g.index.intersection(ph_g.index)
diff_g = pp_g.loc[common_g] - ph_g.loc[common_g]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(diff_g.values, bins=50, color='#3498db', edgecolor='white', linewidth=0.4)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.axvline(diff_g.mean(), color='#e74c3c', linewidth=1.5, label=f'mean = {diff_g.mean():.4f}')
ax.set_xlabel('global_sim(persuasive) − global_sim(hard_negative)')
ax.set_ylabel('Count')
ax.set_title('Paired Difference: Persuasive vs Hard Negative (same post)')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'paired_difference_hist.png', bbox_inches='tight')
plt.show()

## 8. Multi-Feature Classification

Can we predict persuasive vs hard_negative using all 31 TOM features?
We use balanced class weights to handle the 1:2.7 class imbalance.

In [ ]:
# Binary classification: persuasive vs hard_negative
clf_df = df[df['comment_type'].isin(['persuasive', 'hard_negative'])].dropna(subset=all_feature_cols)
X = clf_df[all_feature_cols].values
y = (clf_df['comment_type'] == 'persuasive').astype(int).values
print(f"N={len(X)}, persuasive={y.sum()}, hard_neg={(y==0).sum()}")

X_scaled = StandardScaler().fit_transform(X)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sw = compute_sample_weight('balanced', y)

print("\n--- Binary: persuasive vs hard_negative ---")
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_auc = cross_val_score(lr, X_scaled, y, cv=skf, scoring='roc_auc')
print(f"Logistic Regression  AUC: {lr_auc.mean():.4f} ± {lr_auc.std():.4f}")

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_auc = cross_val_score(rf, X_scaled, y, cv=skf, scoring='roc_auc')
print(f"Random Forest        AUC: {rf_auc.mean():.4f} ± {rf_auc.std():.4f}")

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
gb_cv = cross_validate(gb, X_scaled, y, cv=skf, scoring='roc_auc', fit_params={'sample_weight': sw})
print(f"Gradient Boosting    AUC: {gb_cv['test_score'].mean():.4f} ± {gb_cv['test_score'].std():.4f}")

# 3-way classification
print("\n--- Three-way: persuasive vs hard_neg vs easy_neg ---")
clf3_df = df.dropna(subset=all_feature_cols)
X3 = StandardScaler().fit_transform(clf3_df[all_feature_cols].values)
y3 = clf3_df['comment_type'].values

lr3 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, multi_class='multinomial')
lr3_f1 = cross_val_score(lr3, X3, y3, cv=skf, scoring='f1_macro')
print(f"LR macro-F1: {lr3_f1.mean():.4f} ± {lr3_f1.std():.4f}")

rf3 = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf3_f1 = cross_val_score(rf3, X3, y3, cv=skf, scoring='f1_macro')
print(f"RF macro-F1: {rf3_f1.mean():.4f} ± {rf3_f1.std():.4f}")

## 9. Feature Importance

Which TOM features matter most for predicting persuasion?
We compare coefficients from Logistic Regression (direction), and importances from RF and GB (magnitude).

In [ ]:
# Fit on full data for importance analysis
lr.fit(X_scaled, y)
rf.fit(X_scaled, y)
gb.fit(X_scaled, y, sample_weight=sw)

imp_df = pd.DataFrame({
    'feature': all_feature_cols,
    'lr_coef': lr.coef_[0],
    'rf_imp': rf.feature_importances_,
    'gb_imp': gb.feature_importances_,
})
imp_df['avg_rank'] = (
    imp_df['lr_coef'].abs().rank(ascending=False) +
    imp_df['rf_imp'].rank(ascending=False) +
    imp_df['gb_imp'].rank(ascending=False)
) / 3
imp_df = imp_df.sort_values('avg_rank')
imp_df.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)

print(f"{'Feature':<28} {'LR coef':>9} {'RF imp':>9} {'GB imp':>9} {'Avg Rank':>9}")
print('-' * 68)
for _, r in imp_df.iterrows():
    print(f"{r['feature']:<28} {r['lr_coef']:>9.4f} {r['rf_imp']:>9.4f} {r['gb_imp']:>9.4f} {r['avg_rank']:>9.1f}")

# Plot top features by LR coefficient (shows direction)
top_lr = imp_df.nlargest(10, 'avg_rank', keep='last').sort_values('lr_coef')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_lr['feature'], top_lr['lr_coef'], color=['#e74c3c' if v < 0 else '#2ecc71' for v in top_lr['lr_coef']])
axes[0].set_xlabel('LR Coefficient')
axes[0].set_title('Top Features: Logistic Regression\n(positive = helps persuasion)')
axes[0].axvline(0, color='black', linewidth=0.5)

top_gb = imp_df.nlargest(10, 'gb_imp')
axes[1].barh(top_gb['feature'], top_gb['gb_imp'], color='#3498db')
axes[1].set_xlabel('GB Feature Importance')
axes[1].set_title('Top Features: Gradient Boosting')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_importance.png', bbox_inches='tight')
plt.show()

## 10. TOM Richness & Save Results

In [ ]:
# TOM Richness
comment_richness = []
for p in pairs:
    comment_richness.append({
        'comment_id': p['comment_id'],
        'comment_type': p['comment_type'],
        'richness': sum(1 for cat in CATEGORIES if p['comment_values'].get(cat))
    })
cr_df = pd.DataFrame(comment_richness)

print("Mean TOM category richness (non-null categories) per comment type:")
print(cr_df.groupby('comment_type')['richness'].describe().round(3))

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=cr_df, x='comment_type', y='richness', order=type_order, palette=palette, ax=ax)
ax.set_title('TOM Richness (# non-null categories) by Comment Type')
ax.set_xlabel('')
ax.set_ylabel('# TOM Categories with Values')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tom_richness.png', bbox_inches='tight')
plt.show()

# Save all features
df.to_csv(RESULTS_DIR / 'tom_features.csv', index=False)
print(f"\nSaved {len(df)} rows x {len(all_feature_cols)} features to results/tom_features.csv")